# The CHSH game
Compare every deterministic classical strategy with an ideal Bell-pair strategy. Predict before running each cell. This is a simulator experiment, not a hardware performance benchmark.


In [ ]:
%pip -q install "qiskit>=2,<3"


In [ ]:
import itertools
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector


## 1. Enumerate all local classical strategies
Each tuple gives a0, a1, b0, b1. Inputs are independent and uniform; players cannot communicate during a round.


In [ ]:
inputs = list(itertools.product([0, 1], repeat=2))
strategies = list(itertools.product([0, 1], repeat=4))
def classical_score(s):
    a0, a1, b0, b1 = s
    a, b = [a0, a1], [b0, b1]
    return sum((a[x] ^ b[y]) == (x & y) for x, y in inputs) / 4
scores = [classical_score(s) for s in strategies]
assert max(scores) == 0.75
print("Best classical probability:", max(scores))


## 2. Prepare the Bell pair
Qubit 0 belongs to Alice, qubit 1 to Bob. Qiskit basis strings are ordered q1 q0, so the printed left bit is Bob’s.


In [ ]:
bell = QuantumCircuit(2)
bell.h(0)
bell.cx(0, 1)
print(bell.draw(output="text"))
print(Statevector.from_instruction(bell))


## 3. Choose the measurement bases
A measurement along cos(t) Z + sin(t) X is implemented by Ry(-t) followed by Z readout. Alice uses angles 0 or pi/2; Bob uses pi/4 or -pi/4. We calculate exact output probabilities first.


In [ ]:
def quantum_probs(x, y):
    circuit = bell.copy()
    circuit.ry(-[0, np.pi / 2][x], 0)
    circuit.ry(-[np.pi / 4, -np.pi / 4][y], 1)
    return Statevector.from_instruction(circuit).probabilities_dict()
win_probs = []
for x, y in inputs:
    probs = quantum_probs(x, y)
    win = sum(p for bits, p in probs.items()
              if (int(bits[1]) ^ int(bits[0])) == (x & y))
    win_probs.append(win)
    print((x, y), "win probability", round(win, 6))
ideal = (2 + np.sqrt(2)) / 4
assert np.allclose(win_probs, ideal)
print("Quantum average:", np.mean(win_probs))


## 4. Sample rounds and vary visibility
We mix Bell-state statistics with uniform two-qubit outcomes: rho = v |Phi+><Phi+| + (1-v) I/4. This simple noise model is not a model of every device. Set v to 0, 0.7 and 1. Does the fixed quantum strategy beat 75%?


In [ ]:
rng = np.random.default_rng(42)
visibility = 1.0
rounds = 4000
assert 0 <= visibility <= 1
wins = 0
for _ in range(rounds):
    x, y = inputs[rng.integers(4)]
    exact = quantum_probs(x, y)
    labels = ["00", "01", "10", "11"]
    probs = [visibility * exact.get(k, 0) + (1 - visibility) / 4 for k in labels]
    bits = rng.choice(labels, p=probs)
    wins += (int(bits[1]) ^ int(bits[0])) == (x & y)
expected = 0.5 + visibility * np.sqrt(2) / 4
print("Expected:", expected, "Observed:", wins / rounds)


## Exercises
1. Explain the XOR contradiction that bounds all deterministic strategies.
2. Why does shared randomness not improve that bound?
3. Derive the visibility threshold 1/sqrt(2) for this fixed strategy.
4. Why is this correlation advantage not evidence that every quantum learning model is better?
5. Repeat with different seeds. Why need a finite batch not equal its expectation?

References: [CHSH (1969)](https://doi.org/10.1103/PhysRevLett.23.880); [Qiskit Statevector documentation](https://quantum.cloud.ibm.com/docs/api/qiskit/qiskit.quantum_info.Statevector).
